In [ ]:
# CELL 1 — Install Dependencies

import subprocess, sys

packages = [
    "sentence-transformers",
    "faiss-cpu",
    "fastapi",
    "uvicorn[standard]",
    "nest-asyncio",
    "pydantic",
    "numpy",
    "httpx",
    "tabulate",
]

print("📦 Menginstall dependencies...\n")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\n🎉 Semua dependency berhasil diinstall!")
print("⚠️  Jika ini pertama kali, RESTART KERNEL sebelum lanjut ke cell berikutnya.")

In [ ]:
# CELL 2 — Imports & Konfigurasi Global

import json
import os
import warnings
import numpy as np
from pathlib import Path
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, field
from collections import Counter

warnings.filterwarnings("ignore")

# ── Konfigurasi Path ─────────────────────────────────────────
BASE_DIR            = Path(os.getcwd())
PRODUCTS_DATASET_PATH = BASE_DIR / "data" / "products.json"
ORDERS_DATASET_PATH   = BASE_DIR / "data" / "orders.json"
DATASET_PATH        = PRODUCTS_DATASET_PATH  # Backward compatibility alias

# ── Konfigurasi Model ────────────────────────────────────────
MODEL_NAME    = "BAAI/bge-m3"
TOP_N_DEFAULT = 5

# ── Konfigurasi Hybrid Scoring ───────────────────────────────
COMPLEMENTARY_WEIGHT = 0.40   # Bobot skor komplementer kategori
SEMANTIC_WEIGHT      = 0.60   # Bobot cosine similarity embedding

print("✅ Imports berhasil.")
print(f"   Base Dir       : {BASE_DIR}")
print(f"   Products Path  : {PRODUCTS_DATASET_PATH}")
print(f"   Orders Path    : {ORDERS_DATASET_PATH}")
print(f"   Model          : {MODEL_NAME}")
print(f"   Hybrid Score   : {COMPLEMENTARY_WEIGHT*100:.0f}% Complementary + {SEMANTIC_WEIGHT*100:.0f}% Semantic")


In [ ]:
# CELL 3 — Load Dataset (Products & Orders)

@dataclass
class Product:
    """
    Representasi satu produk UMKM.
    Field-field disesuaikan dengan format data fullstack Tenantify.
    """
    id          : str
    tenantId    : str
    tenantName  : str
    boothNumber : str
    name        : str
    category    : str
    price       : int
    description : str
    rating      : float
    image       : str
    soldCount   : int
    isPopular   : bool

    @property
    def text_for_embedding(self) -> str:
        """
        Menggabungkan seluruh informasi produk menjadi satu string
        kaya konteks untuk diproses model embedding.
        """
        return (
            f"{self.name}. "
            f"{self.description} "
            f"Kategori: {self.category}. "
            f"Dari toko: {self.tenantName}."
        )


@dataclass
class OrderItem:
    """Representasi satu item dalam pesanan pelanggan."""
    productName : str
    tenantName  : str
    price       : int
    quantity    : int
    notes       : str = ""


@dataclass
class Order:
    """
    Representasi pesanan pelanggan.
    Field-field disesuaikan dengan format pesanan pelanggan fullstack Tenantify.
    """
    id           : str
    pickupCode   : str
    orderType    : str
    tableNumber  : Optional[str]
    tenantNames  : List[str]
    items        : List[OrderItem]
    subtotal     : int
    platformFee  : int
    total        : int
    status       : str
    paymentMethod: str
    createdAt    : str


def load_products(path: Path) -> List[Product]:
    """Membaca products.json dan mengkonversi ke list of Product."""
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset produk tidak ditemukan di: {path}\n"
            "Pastikan file data/products.json ada di direktori proyek."
        )
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    return [Product(**item) for item in raw]


def load_orders(path: Path) -> List[Order]:
    """Membaca orders.json dan mengkonversi ke list of Order."""
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset pesanan tidak ditemukan di: {path}\n"
            "Pastikan file data/orders.json ada di direktori proyek."
        )
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    orders = []
    for item in raw:
        items = [OrderItem(**it) for it in item["items"]]
        order_dict = {**item, "items": items}
        orders.append(Order(**order_dict))
    return orders


# Alias backward compatibility
load_dataset = load_products

# Load kedua dataset
products = load_products(PRODUCTS_DATASET_PATH)
orders   = load_orders(ORDERS_DATASET_PATH)

# Ringkasan dataset produk
category_counts = Counter(p.category for p in products)
tenant_counts   = Counter(p.tenantId for p in products)

# Ringkasan dataset orders
order_type_counts = Counter(o.orderType for o in orders)
order_status_counts = Counter(o.status for o in orders)
payment_counts = Counter(o.paymentMethod for o in orders)

print(f"📦 KEDUA DATASET BERHASIL DIMUAT!")
print(f"   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"   [1] DATASET PRODUK SELLER (products.json):")
print(f"       • Total Produk : {len(products)}")
print(f"       • Total Tenant : {len(tenant_counts)}")
print(f"       • Kategori     :")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"         - {cat:<12} {bar} ({count})")

print(f"
   [2] DATASET PESANAN PELANGGAN (orders.json):")
print(f"       • Total Pesanan: {len(orders)}")
print(f"       • Tipe Order   : {dict(order_type_counts)}")
print(f"       • Status       : {dict(order_status_counts)}")
print(f"       • Metode Bayar : {dict(payment_counts)}")

print(f"
🏪 Contoh Produk:")
for p in products[:2]:
    print(f"   • [{p.id}] {p.name} → {p.tenantName} ({p.boothNumber}) | Rp{p.price:,} | ⭐{p.rating}")

print(f"
🧾 Contoh Pesanan Pelanggan:")
for o in orders[:2]:
    items_desc = ", ".join(f"{it.productName} x{it.quantity}" for it in o.items)
    print(f"   • [{o.id}] {o.pickupCode} | {o.orderType} ({o.tableNumber or 'Takeaway'}) | Rp{o.total:,} [{o.status}]")
    print(f"     Items: {items_desc}")


In [ ]:
# ============================================================
# CELL 4 — Embedder: Wrapper BAAI/bge-m3
# ⚠️ Download model ~2.3GB pada pertama kali dijalankan.
# ============================================================

import faiss
from sentence_transformers import SentenceTransformer


class Embedder:
    """
    Wrapper untuk model BAAI/bge-m3 (multilingual, support Bahasa Indonesia).
    Tugasnya:
      1. Load model dari HuggingFace (sekali saja)
      2. Encode teks menjadi dense vector
      3. Precompute & simpan embedding semua produk
      4. Build FAISS index untuk pencarian cepat
    """

    def __init__(self, model_name: str = MODEL_NAME):
        print(f"⏳ Memuat model {model_name}...")
        print("   (Download ~2.3GB jika pertama kali — harap sabar)\n")
        self.model = SentenceTransformer(model_name)
        self.embeddings: Optional[np.ndarray] = None
        self.index: Optional[faiss.IndexFlatIP] = None
        print(f"✅ Model berhasil dimuat!")
        print(f"   Embedding Dim : {self.model.get_sentence_embedding_dimension()}")

    def encode(self, texts: List[str], batch_size: int = 32, show_progress: bool = True) -> np.ndarray:
        """Encode list of teks → numpy array of L2-normalized embeddings."""
        vecs = self.model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,   # L2-normalize → cosine sim = inner product
            show_progress_bar=show_progress,
        )
        return vecs.astype(np.float32)

    def build_index(self, product_list: List[Product]) -> None:
        """
        Precompute embedding semua produk dan build FAISS index.
        Dipanggil sekali saat inisialisasi sistem.
        """
        print(f"\n🔄 Membangun index untuk {len(product_list)} produk...")
        texts = [p.text_for_embedding for p in product_list]
        self.embeddings = self.encode(texts, show_progress=True)

        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)   # Inner Product index
        self.index.add(self.embeddings)

        print(f"✅ FAISS index selesai dibangun!")
        print(f"   Vektor tersimpan : {self.index.ntotal}")
        print(f"   Dimensi          : {dim}")

    def get_similarities(self, query_idx: int) -> np.ndarray:
        """
        Hitung cosine similarity antara produk di query_idx
        dengan semua produk lain di index.
        """
        if self.index is None:
            raise RuntimeError("Index belum dibangun. Panggil build_index() terlebih dahulu.")
        query_vec = self.embeddings[query_idx:query_idx + 1]
        scores, _ = self.index.search(query_vec, self.index.ntotal)
        return scores[0]

    def encode_query(self, text: str) -> np.ndarray:
        """Encode satu teks query untuk pencarian ad-hoc."""
        return self.encode([text], show_progress=False)[0]

    def get_similarities_from_vector(self, query_vec: np.ndarray) -> np.ndarray:
        """Hitung similarity dari vector yang sudah di-encode."""
        scores, _ = self.index.search(query_vec.reshape(1, -1), self.index.ntotal)
        return scores[0]


# Inisialisasi Embedder & build index
embedder = Embedder(model_name=MODEL_NAME)
embedder.build_index(products)

In [ ]:
# ============================================================
# CELL 5 — Recommender: Core Hybrid Scoring Engine
# ============================================================

# ── Complementary Category Map ───────────────────────────────
# Kategori mengikuti format fullstack: Minuman, Makanan, Camilan, Gorengan, Dessert
# Score 1.0 = sangat komplementer | 0.0 = tidak relevan
COMPLEMENTARY_MAP: Dict[tuple, float] = {
    # Camilan ↔ Minuman (paling cocok)
    ("Camilan",  "Minuman")  : 1.0,
    ("Camilan",  "Dessert")  : 0.6,
    ("Camilan",  "Gorengan") : 0.3,

    # Gorengan ↔ Minuman
    ("Gorengan", "Minuman")  : 1.0,
    ("Gorengan", "Dessert")  : 0.5,
    ("Gorengan", "Camilan")  : 0.3,

    # Makanan ↔ Minuman & Dessert
    ("Makanan",  "Minuman")  : 0.9,
    ("Makanan",  "Dessert")  : 0.8,
    ("Makanan",  "Camilan")  : 0.4,
    ("Makanan",  "Gorengan") : 0.3,

    # Dessert ↔ Minuman
    ("Dessert",  "Minuman")  : 1.0,
    ("Dessert",  "Camilan")  : 0.5,

    # Minuman ↔ Minuman (kompetitif, bukan komplementer)
    ("Minuman",  "Minuman")  : 0.15,
}


def get_complementary_score(cat_a: str, cat_b: str) -> float:
    """Ambil skor komplementer antara dua kategori (dua arah)."""
    if cat_a == cat_b:
        # Kategori sama: sedikit relevan tapi bukan prioritas
        return COMPLEMENTARY_MAP.get((cat_a, cat_b), 0.2)
    score = COMPLEMENTARY_MAP.get((cat_a, cat_b))
    if score is None:
        score = COMPLEMENTARY_MAP.get((cat_b, cat_a), 0.1)
    return score


@dataclass
class RecommendationResult:
    """Satu item hasil rekomendasi beserta skor dan penjelasannya."""
    product             : Product
    hybrid_score        : float
    semantic_score      : float
    complementary_score : float
    reason              : str


class Recommender:
    """
    Core Recommendation Engine untuk Tenantify.

    hybrid_score = (COMPLEMENTARY_WEIGHT × complementary_score)
                 + (SEMANTIC_WEIGHT      × semantic_similarity)

    Constraint utama:
      - Produk dari tenantId BERBEDA (pemerataan pedagang)
    """

    def __init__(self, product_list: List[Product], embedder: Embedder):
        self.products     = product_list
        self.embedder     = embedder
        # Mapping id → index untuk lookup O(1)
        self.id_to_idx    = {p.id: i for i, p in enumerate(product_list)}
        # Mapping name → product untuk lookup dari order items
        self.name_to_product: Dict[str, Product] = {p.name: p for p in product_list}
        print("✅ Recommender siap digunakan.")

    def recommend(
        self,
        product_id: str,
        top_n: int = TOP_N_DEFAULT,
        exclude_same_tenant: bool = True,
        category_filter: Optional[str] = None,
        exclude_product_ids: Optional[List[str]] = None,
    ) -> List[RecommendationResult]:
        """
        Rekomendasi produk komplementer berdasarkan satu produk.

        Args:
            product_id          : ID produk yang dipilih user (format: 'cp-1', 'ag-1', dst)
            top_n               : Jumlah rekomendasi yang dikembalikan
            exclude_same_tenant : Jika True, tidak rekomendasikan dari tenant yang sama
            category_filter     : Jika diisi, hanya rekomendasikan dari kategori ini
            exclude_product_ids : Daftar product_id yang dikecualikan dari hasil
        """
        if product_id not in self.id_to_idx:
            raise ValueError(f"Produk '{product_id}' tidak ditemukan di dataset.")

        query_idx     = self.id_to_idx[product_id]
        query_product = self.products[query_idx]
        exclude_ids   = set(exclude_product_ids or [])
        exclude_ids.add(product_id)  # Selalu exclude diri sendiri

        sim_scores = self.embedder.get_similarities(query_idx)

        results: List[RecommendationResult] = []
        for idx, candidate in enumerate(self.products):
            if candidate.id in exclude_ids:
                continue
            if exclude_same_tenant and candidate.tenantId == query_product.tenantId:
                continue
            if category_filter and candidate.category != category_filter:
                continue

            comp_score = get_complementary_score(query_product.category, candidate.category)
            sem_score  = float(np.clip(sim_scores[idx], 0.0, 1.0))
            hybrid     = COMPLEMENTARY_WEIGHT * comp_score + SEMANTIC_WEIGHT * sem_score
            reason     = _build_reason(query_product, candidate, comp_score, sem_score)

            results.append(RecommendationResult(
                product             = candidate,
                hybrid_score        = round(hybrid, 4),
                semantic_score      = round(sem_score, 4),
                complementary_score = round(comp_score, 4),
                reason              = reason,
            ))

        results.sort(key=lambda r: r.hybrid_score, reverse=True)
        return results[:top_n]

    def recommend_from_order(
        self,
        order_items: List[Dict[str, Any]],
        top_n: int = TOP_N_DEFAULT,
    ) -> List[RecommendationResult]:
        """
        Rekomendasi berbasis pesanan aktif (order items dari fullstack).
        Mengagregasi embedding semua item yang sudah dipesan,
        lalu mencari produk komplementer yang belum ada di pesanan.

        Args:
            order_items : List item dari field 'items' pada order object fullstack
                          Format: [{ productName, tenantName, price, quantity, notes }, ...]
            top_n       : Jumlah rekomendasi yang dikembalikan
        """
        # Resolve order items ke Product objects
        ordered_products: List[Product] = []
        for item in order_items:
            product = self.name_to_product.get(item["productName"])
            if product:
                ordered_products.append(product)

        if not ordered_products:
            raise ValueError("Tidak ada produk dari order yang cocok dengan dataset.")

        # Kumpulkan semua tenantId & productId yang sudah ada di order
        ordered_tenant_ids  = {p.tenantId  for p in ordered_products}
        ordered_product_ids = {p.id        for p in ordered_products}

        # Agregasi embedding: rata-rata embedding semua produk yang dipesan
        ordered_idxs   = [self.id_to_idx[p.id] for p in ordered_products]
        ordered_vecs   = self.embedder.embeddings[ordered_idxs]
        mean_vec       = ordered_vecs.mean(axis=0)
        # L2-normalize ulang setelah rata-rata
        norm           = np.linalg.norm(mean_vec)
        mean_vec       = (mean_vec / norm).astype(np.float32)

        # Hitung similarity dari mean vector ke semua produk
        sim_scores = self.embedder.get_similarities_from_vector(mean_vec)

        # Kategori dominan dari order untuk complementary scoring
        category_counts = Counter(p.category for p in ordered_products)
        dominant_category = category_counts.most_common(1)[0][0]

        results: List[RecommendationResult] = []
        for idx, candidate in enumerate(self.products):
            # Skip produk yang sudah ada di order
            if candidate.id in ordered_product_ids:
                continue
            # Skip tenant yang sudah ada di order (pemerataan)
            if candidate.tenantId in ordered_tenant_ids:
                continue

            comp_score = get_complementary_score(dominant_category, candidate.category)
            sem_score  = float(np.clip(sim_scores[idx], 0.0, 1.0))
            hybrid     = COMPLEMENTARY_WEIGHT * comp_score + SEMANTIC_WEIGHT * sem_score

            reason = (
                f"{candidate.category} ini cocok melengkapi pesananmu "
                f"yang didominasi {dominant_category} "
                f"(relevansi: {int(sem_score*100)}%)."
            )

            results.append(RecommendationResult(
                product             = candidate,
                hybrid_score        = round(hybrid, 4),
                semantic_score      = round(sem_score, 4),
                complementary_score = round(comp_score, 4),
                reason              = reason,
            ))

        results.sort(key=lambda r: r.hybrid_score, reverse=True)
        return results[:top_n]


def _build_reason(query: Product, candidate: Product, comp: float, sem: float) -> str:
    """Buat kalimat penjelasan singkat mengapa produk ini direkomendasikan."""
    comp_label = (
        "sangat cocok dipadukan" if comp >= 0.9 else
        "cocok dipadukan"        if comp >= 0.7 else
        "cukup cocok dipadukan"  if comp >= 0.5 else
        "bisa dipadukan"
    )
    return (
        f"{candidate.category} ini {comp_label} dengan "
        f"{query.category} '{query.name}' "
        f"(relevansi konteks: {int(sem*100)}%)."
    )


# Inisialisasi Recommender
recommender = Recommender(product_list=products, embedder=embedder)

In [ ]:
# ============================================================
# CELL 6 — Pydantic Schemas (Request & Response)
# Format disesuaikan dengan data fullstack Tenantify
# ============================================================

from pydantic import BaseModel, Field


# ── Product Schema (sama persis dengan format fullstack) ─────
class ProductSchema(BaseModel):
    id          : str
    tenantId    : str
    tenantName  : str
    boothNumber : str
    name        : str
    category    : str
    price       : int
    description : str
    rating      : float
    image       : str
    soldCount   : int
    isPopular   : bool


# ── Order Item Schema (dari field 'items' di order object) ───
class OrderItemSchema(BaseModel):
    productName : str  = Field(..., description="Nama produk")
    tenantName  : str  = Field(..., description="Nama tenant")
    price       : int  = Field(..., description="Harga satuan")
    quantity    : int  = Field(1,   description="Jumlah")
    notes       : str  = Field("",  description="Catatan dari pembeli")


# ── Order Schema (sama persis dengan format fullstack) ───────
class OrderSchema(BaseModel):
    id           : str
    pickupCode   : str
    orderType    : str
    tableNumber  : Optional[str] = None
    tenantNames  : List[str]
    items        : List[OrderItemSchema]
    subtotal     : int
    platformFee  : int
    total        : int
    status       : str
    paymentMethod: str
    createdAt    : str


# ── Recommendation Item ──────────────────────────────────────
class RecommendationItem(BaseModel):
    product             : ProductSchema
    hybrid_score        : float = Field(description="Skor gabungan [0–1]")
    semantic_score      : float = Field(description="Cosine similarity embedding [0–1]")
    complementary_score : float = Field(description="Skor kesesuaian kategori [0–1]")
    reason              : str   = Field(description="Alasan rekomendasi")


# ── Request Bodies ───────────────────────────────────────────
class RecommendByProductRequest(BaseModel):
    """Request rekomendasi berdasarkan satu produk."""
    product_id           : str           = Field(...,  description="ID produk yang dipilih", example="gk-1")
    top_n                : int           = Field(5,    ge=1, le=20)
    exclude_same_tenant  : bool          = Field(True, description="Exclude produk dari tenant yang sama")
    category_filter      : Optional[str] = Field(None, description="Filter kategori: Minuman/Makanan/Camilan/Gorengan/Dessert")


class RecommendByOrderRequest(BaseModel):
    """Request rekomendasi berdasarkan pesanan aktif (order items)."""
    items : List[OrderItemSchema] = Field(..., description="List item pesanan dari order object")
    top_n : int                   = Field(5,   ge=1, le=20)


# ── Response Bodies ──────────────────────────────────────────
class RecommendByProductResponse(BaseModel):
    query_product   : ProductSchema
    recommendations : List[RecommendationItem]
    total_returned  : int


class RecommendByOrderResponse(BaseModel):
    ordered_products : List[ProductSchema]
    recommendations  : List[RecommendationItem]
    total_returned   : int


class HealthResponse(BaseModel):
    status        : str
    model         : str
    total_products: int
    total_tenants : int
    total_orders  : int


# ── Serializer Helpers ───────────────────────────────────────
def product_to_schema(p: Product) -> ProductSchema:
    return ProductSchema(
        id=p.id, tenantId=p.tenantId, tenantName=p.tenantName,
        boothNumber=p.boothNumber, name=p.name, category=p.category,
        price=p.price, description=p.description, rating=p.rating,
        image=p.image, soldCount=p.soldCount, isPopular=p.isPopular
    )


def order_to_schema(o: Order) -> OrderSchema:
    return OrderSchema(
        id=o.id,
        pickupCode=o.pickupCode,
        orderType=o.orderType,
        tableNumber=o.tableNumber,
        tenantNames=o.tenantNames,
        items=[
            OrderItemSchema(
                productName=it.productName,
                tenantName=it.tenantName,
                price=it.price,
                quantity=it.quantity,
                notes=it.notes or ""
            ) for it in o.items
        ],
        subtotal=o.subtotal,
        platformFee=o.platformFee,
        total=o.total,
        status=o.status,
        paymentMethod=o.paymentMethod,
        createdAt=o.createdAt
    )


print("✅ Pydantic schemas berhasil didefinisikan.")
print("   Schemas yang tersedia:")
print("     • ProductSchema             (sama persis dengan format produk fullstack)")
print("     • OrderItemSchema           (format item dalam pesanan)")
print("     • OrderSchema               (sama persis dengan format order fullstack)")
print("     • RecommendByProductRequest / Response")
print("     • RecommendByOrderRequest   / Response")
print("     • HealthResponse")


In [ ]:
# ============================================================
# CELL 7 — FastAPI App + Jalankan Server di Notebook

import nest_asyncio
import uvicorn
import threading
import time
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware

# Patch event loop agar FastAPI bisa jalan di dalam Jupyter
nest_asyncio.apply()

# ── Inisialisasi FastAPI ─────────────────────────────────────
app = FastAPI(
    title       = "Tenantify AI — Recommendation API",
    description = (
        "Sistem rekomendasi produk UMKM berbasis Content-Based Filtering "
        "menggunakan model `BAAI/bge-m3`. Dirancang untuk pemerataan peluang tenant UMKM."
    ),
    version  = "1.1.0",
    docs_url = "/docs",
    redoc_url= "/redoc",
)

# ── CORS (untuk integrasi frontend fullstack) ────────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins  = ["*"],
    allow_methods  = ["*"],
    allow_headers  = ["*"],
)


# ── Endpoints ────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse, tags=["System"])
def health_check():
    """Cek status server, model, dan statistik kedua dataset."""
    tenant_ids = {p.tenantId for p in products}
    return HealthResponse(
        status        = "ok",
        model         = MODEL_NAME,
        total_products= len(products),
        total_tenants = len(tenant_ids),
        total_orders  = len(orders),
    )


@app.get("/products", response_model=List[ProductSchema], tags=["Products"])
def list_products(
    category  : Optional[str] = None,
    tenant_id : Optional[str] = None,
    is_popular: Optional[bool] = None,
):
    """
    Ambil semua produk. Bisa difilter berdasarkan:
    - **category**: Minuman | Makanan | Camilan | Gorengan | Dessert
    - **tenant_id**: ID tenant (misal: tenant-1)
    - **is_popular**: true / false
    """
    filtered = products
    if category:
        filtered = [p for p in filtered if p.category == category]
    if tenant_id:
        filtered = [p for p in filtered if p.tenantId == tenant_id]
    if is_popular is not None:
        filtered = [p for p in filtered if p.isPopular == is_popular]
    return [product_to_schema(p) for p in filtered]


@app.get("/products/{product_id}", response_model=ProductSchema, tags=["Products"])
def get_product(product_id: str):
    """Ambil detail satu produk berdasarkan ID (contoh: `gk-1`, `cp-1`, `ag-1`)."""
    idx = recommender.id_to_idx.get(product_id)
    if idx is None:
        raise HTTPException(status_code=404, detail=f"Produk '{product_id}' tidak ditemukan.")
    return product_to_schema(products[idx])


@app.get("/orders", response_model=List[OrderSchema], tags=["Orders"])
def list_orders(
    status    : Optional[str] = None,
    order_type: Optional[str] = None,
):
    """
    Ambil daftar pesanan pelanggan dari dataset `orders.json`.
    Bisa difilter berdasarkan:
    - **status**: Siap Diambil | Sedang Diproses | Selesai
    - **order_type**: Dine-in | Takeaway
    """
    filtered = orders
    if status:
        filtered = [o for o in filtered if o.status.lower() == status.lower()]
    if order_type:
        filtered = [o for o in filtered if order_type.lower() in o.orderType.lower()]
    return [order_to_schema(o) for o in filtered]


@app.get("/orders/{order_id}", response_model=OrderSchema, tags=["Orders"])
def get_order(order_id: str):
    """Ambil detail satu pesanan pelanggan berdasarkan Order ID (contoh: `ORD-9201`)."""
    for o in orders:
        if o.id == order_id:
            return order_to_schema(o)
    raise HTTPException(status_code=404, detail=f"Pesanan '{order_id}' tidak ditemukan.")


@app.post("/recommend/by-product", response_model=RecommendByProductResponse, tags=["Recommendation"])
def recommend_by_product(req: RecommendByProductRequest):
    """
    **Rekomendasi berdasarkan produk yang dipilih.**

    Gunakan ketika user sedang melihat detail produk dan ingin
    menampilkan seksi *'Produk lain yang cocok dinikmati bersama'*.

    **Contoh request:**
    ```json
    { "product_id": "gk-1", "top_n": 5, "exclude_same_tenant": true }
    ```
    """
    try:
        results = recommender.recommend(
            product_id          = req.product_id,
            top_n               = req.top_n,
            exclude_same_tenant = req.exclude_same_tenant,
            category_filter     = req.category_filter,
        )
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

    query_idx = recommender.id_to_idx[req.product_id]
    return RecommendByProductResponse(
        query_product   = product_to_schema(products[query_idx]),
        recommendations = [
            RecommendationItem(
                product             = product_to_schema(r.product),
                hybrid_score        = r.hybrid_score,
                semantic_score      = r.semantic_score,
                complementary_score = r.complementary_score,
                reason              = r.reason,
            ) for r in results
        ],
        total_returned = len(results),
    )


@app.post("/recommend/by-order", response_model=RecommendByOrderResponse, tags=["Recommendation"])
def recommend_by_order(req: RecommendByOrderRequest):
    """
    **Rekomendasi berdasarkan pesanan aktif (order items).**

    Gunakan ketika user sudah ada pesanan aktif dan ingin
    menampilkan seksi *'Tambahkan ke pesananmu'*.
    Menerima langsung field `items` dari order object fullstack.

    **Contoh request (langsung pakai items dari order object):**
    ```json
    {
      "items": [
        { "productName": "Kopi Susu Gula Aren", "tenantName": "Kopi & Roti Nusantara",
          "price": 18000, "quantity": 1, "notes": "" },
        { "productName": "Paket Ayam Geprek Krispi + Nasi", "tenantName": "Ayam Geprek Sambal Korek",
          "price": 25000, "quantity": 1, "notes": "Level pedas sedang" }
      ],
      "top_n": 5
    }
    ```
    """
    try:
        results = recommender.recommend_from_order(
            order_items = [item.model_dump() for item in req.items],
            top_n       = req.top_n,
        )
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    ordered_products = [
        p for item in req.items
        if (p := recommender.name_to_product.get(item.productName))
    ]

    return RecommendByOrderResponse(
        ordered_products = [product_to_schema(p) for p in ordered_products],
        recommendations  = [
            RecommendationItem(
                product             = product_to_schema(r.product),
                hybrid_score        = r.hybrid_score,
                semantic_score      = r.semantic_score,
                complementary_score = r.complementary_score,
                reason              = r.reason,
            ) for r in results
        ],
        total_returned = len(results),
    )


@app.post("/recommend/by-order-id/{order_id}", response_model=RecommendByOrderResponse, tags=["Recommendation"])
def recommend_by_order_id(order_id: str, top_n: int = Query(5, ge=1, le=20)):
    """
    **Rekomendasi langsung berdasarkan ID pesanan yang tersimpan.**

    Sangat praktis untuk frontend: cukup kirim `order_id` (contoh: `ORD-9201` atau `ORD-9205`).
    AI otomatis mengambil item-item dari pesanan tersebut dan menghasilkan rekomendasi komplementer
    dari tenant lain.
    """
    order = next((o for o in orders if o.id == order_id), None)
    if not order:
        raise HTTPException(status_code=404, detail=f"Pesanan '{order_id}' tidak ditemukan.")

    order_items = [
        {
            "productName": it.productName,
            "tenantName" : it.tenantName,
            "price"      : it.price,
            "quantity"   : it.quantity,
            "notes"      : it.notes or ""
        }
        for it in order.items
    ]

    try:
        results = recommender.recommend_from_order(
            order_items = order_items,
            top_n       = top_n,
        )
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    ordered_products = [
        p for it in order.items
        if (p := recommender.name_to_product.get(it.productName))
    ]

    return RecommendByOrderResponse(
        ordered_products = [product_to_schema(p) for p in ordered_products],
        recommendations  = [
            RecommendationItem(
                product             = product_to_schema(r.product),
                hybrid_score        = r.hybrid_score,
                semantic_score      = r.semantic_score,
                complementary_score = r.complementary_score,
                reason              = r.reason,
            ) for r in results
        ],
        total_returned = len(results),
    )


# ── Jalankan Server di background thread ─────────────────────
PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Tunggu server siap

print("🚀 Tenantify AI Server berjalan!")
print(f"   Base URL     : http://localhost:{PORT}")
print(f"   Swagger UI   : http://localhost:{PORT}/docs   ← Coba semua endpoint di sini!")
print(f"   ReDoc        : http://localhost:{PORT}/redoc")
print(f"\n   Endpoints:")
print(f"     GET  /health                         → Status server & statistik 2 dataset")
print(f"     GET  /products                       → List semua produk (support filter)")
print(f"     GET  /products/{{id}}                  → Detail produk")
print(f"     GET  /orders                         → List semua pesanan pelanggan")
print(f"     GET  /orders/{{id}}                    → Detail pesanan pelanggan")
print(f"     POST /recommend/by-product           → Rekomendasi dari 1 produk")
print(f"     POST /recommend/by-order             → Rekomendasi dari list items pesanan")
print(f"     POST /recommend/by-order-id/{{order_id}} → Rekomendasi langsung dari ID pesanan")


In [ ]:
# ============================================================
# CELL 8 — Demo: Rekomendasi Berdasarkan Produk
# ============================================================

import httpx
from tabulate import tabulate

BASE_URL = f"http://localhost:{PORT}"


def demo_by_product(product_id: str, top_n: int = 5):
    """Demo endpoint POST /recommend/by-product"""
    resp = httpx.post(f"{BASE_URL}/recommend/by-product", json={
        "product_id"         : product_id,
        "top_n"              : top_n,
        "exclude_same_tenant": True,
    })
    data = resp.json()
    q    = data["query_product"]

    print("=" * 72)
    print(f"🛍️  PRODUK YANG DIPILIH USER")
    print(f"   [{q['id']}] {q['name']}")
    print(f"   Tenant      : {q['tenantName']} ({q['boothNumber']})")
    print(f"   Kategori    : {q['category']} | Harga: Rp{q['price']:,} | ⭐ {q['rating']}")
    print(f"   Deskripsi   : {q['description'][:75]}...")
    print("=" * 72)
    print(f"\n🎯 TOP {top_n} REKOMENDASI PRODUK KOMPLEMENTER:\n")

    rows = []
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        rows.append([
            i, p["id"], p["name"][:28], p["category"],
            f"Rp{p['price']:,}", f"⭐{p['rating']}",
            p["boothNumber"],
            f"{item['hybrid_score']:.3f}",
            f"{item['complementary_score']:.2f}",
            f"{item['semantic_score']:.3f}",
        ])

    print(tabulate(
        rows,
        headers=["#", "ID", "Nama Produk", "Kategori", "Harga",
                 "Rating", "Booth", "Hybrid ▼", "Kompl.", "Semantik"],
        tablefmt="rounded_outline"
    ))

    print(f"\n💡 ALASAN REKOMENDASI:")
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        print(f"   {i}. [{p['id']}] {p['name']}: {item['reason']}")
    print()


# Demo 1: Kue Ikan Isi Coklat (gk-1)
print("━" * 72)
print("DEMO 1 — /recommend/by-product : Kue Ikan Isi Coklat")
print("━" * 72)
demo_by_product("gk-1", top_n=5)

# Demo 2: Paket Ayam Geprek (ag-1)
print("━" * 72)
print("DEMO 2 — /recommend/by-product : Paket Ayam Geprek Krispi + Nasi")
print("━" * 72)
demo_by_product("ag-1", top_n=5)

In [ ]:
# ============================================================
# CELL 9 — Demo: Rekomendasi Berdasarkan Pesanan Aktif
# Menggunakan dataset orders.json asli dari fullstack Tenantify
# ============================================================

def demo_by_order_items(order_obj: dict, top_n: int = 5):
    """Demo endpoint POST /recommend/by-order menggunakan dict pesanan."""
    resp = httpx.post(f"{BASE_URL}/recommend/by-order", json={
        "items": order_obj["items"],
        "top_n": top_n,
    })
    data = resp.json()

    print("=" * 72)
    print(f"🧾 PESANAN AKTIF  [Order ID: {order_obj['id']} | Kode: {order_obj['pickupCode']}]")
    print(f"   {order_obj['orderType']} — {order_obj.get('tableNumber') or 'Takeaway'}")
    for it in order_obj["items"]:
        note = f" (Catatan: {it['notes']})" if it.get('notes') else ""
        print(f"   • {it['productName']} x{it['quantity']}  →  {it['tenantName']}  (Rp{it['price']:,}){note}")
    print(f"   Total: Rp{order_obj['total']:,} | Status: {order_obj['status']}")
    print("=" * 72)
    print(f"\n🎯 TOP {top_n} REKOMENDASI 'TAMBAHKAN KE PESANANMU':\n")

    rows = []
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        rows.append([
            i, p["id"], p["name"][:28], p["category"],
            f"Rp{p['price']:,}", f"⭐{p['rating']}",
            p["tenantName"][:20],
            f"{item['hybrid_score']:.3f}",
        ])

    print(tabulate(
        rows,
        headers=["#", "ID", "Nama Produk", "Kategori", "Harga", "Rating", "Tenant Rekomendasi", "Score ▼"],
        tablefmt="rounded_outline"
    ))

    print(f"\n💡 ALASAN REKOMENDASI:")
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        print(f"   {i}. [{p['id']}] {p['name']}: {item['reason']}")
    print()


def demo_by_order_id(order_id: str, top_n: int = 5):
    """Demo endpoint POST /recommend/by-order-id/{order_id}."""
    resp = httpx.post(f"{BASE_URL}/recommend/by-order-id/{order_id}?top_n={top_n}")
    if resp.status_code != 200:
        print(f"❌ Error {resp.status_code}: {resp.text}")
        return
    data = resp.json()

    # Ambil info order lokal
    order = next((o for o in orders if o.id == order_id), None)
    print("=" * 72)
    print(f"🧾 REKOMENDASI LANGSUNG DARI ORDER ID: {order_id}")
    if order:
        print(f"   {order.orderType} | {order.tableNumber or 'Takeaway'} | Total: Rp{order.total:,}")
        for it in order.items:
            print(f"   • {it.productName} x{it.quantity} dari '{it.tenantName}'")
    print("=" * 72)
    print(f"\n🎯 TOP {top_n} REKOMENDASI UNTUK PESANAN INI:\n")

    rows = []
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        rows.append([
            i, p["id"], p["name"][:28], p["category"],
            f"Rp{p['price']:,}", p["tenantName"][:22],
            f"{item['hybrid_score']:.3f}",
        ])

    print(tabulate(
        rows,
        headers=["#", "ID", "Nama Rekomendasi", "Kategori", "Harga", "Tenant", "Score ▼"],
        tablefmt="rounded_outline"
    ))
    print()


# ── Demo 3: Rekomendasi dari orders[0] (Kopi Susu + Ayam Geprek) ──
print("━" * 72)
print("DEMO 3 — /recommend/by-order : Menggunakan Data ORD-9201 dari orders.json")
print("━" * 72)
sample_order_dict = {
    "id": orders[0].id,
    "pickupCode": orders[0].pickupCode,
    "orderType": orders[0].orderType,
    "tableNumber": orders[0].tableNumber,
    "items": [
        {"productName": it.productName, "tenantName": it.tenantName, "price": it.price, "quantity": it.quantity, "notes": it.notes}
        for it in orders[0].items
    ],
    "total": orders[0].total,
    "status": orders[0].status
}
demo_by_order_items(sample_order_dict, top_n=5)


# ── Demo 4: Rekomendasi langsung dari Order ID ORD-9205 (Kue Ikan Coklat!) ──
print("━" * 72)
print("DEMO 4 — /recommend/by-order-id/ORD-9205 : Pesanan Kue Ikan Isi Coklat")
print("━" * 72)
demo_by_order_id("ORD-9205", top_n=5)


# ── Demo 5: Testing Endpoint Orders ─────────────────────────
print("━" * 72)
print("DEMO 5 — GET /orders : Query Pesanan Pelanggan")
print("━" * 72)
resp = httpx.get(f"{BASE_URL}/orders?status=Siap Diambil")
ready_orders = resp.json()
print(f"📋 Pesanan dengan status 'Siap Diambil' ({len(ready_orders)} pesanan):")
for ro in ready_orders[:3]:
    print(f"   • [{ro['id']}] {ro['pickupCode']} | {ro['orderType']} | Rp{ro['total']:,}")
    for it in ro['items']:
        print(f"     - {it['productName']} x{it['quantity']} ({it['tenantName']})")

print("\n━" * 72)
print("✅ Semua demo selesai!")
print(f"   Swagger UI: http://localhost:{PORT}/docs")
print("━" * 72)
